# NARCISSUS recreation — Colab driver

Run this notebook on Colab (browser **or** the VS Code Google Colab extension) connected to a GPU runtime. The smoke test at the bottom verifies the train + eval pipeline before we invest GPU-hours on the real Figure-3 grid.

**Requires:** Colab Pro, Google account with Drive access.

## 1. Mount Drive

Drive holds the dataset cache, results CSV, and any trigger checkpoints — so a session crash doesn't lose work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/narcissus-recreation'
!mkdir -p {WORK}/data {WORK}/results {WORK}/checkpoints {WORK}/triggers
!ls {WORK}

## 2. Clone (or pull) the repo

The repo is public, so no auth needed. Re-running this cell after pushing changes from your laptop pulls the latest.

In [ ]:
%cd /content
import os
if not os.path.isdir('narcissus-recreation-proposal'):
    !git clone https://github.com/alexzkhan07/narcissus-recreation-proposal.git
else:
    %cd narcissus-recreation-proposal
    !git pull
    %cd /content
%cd narcissus-recreation-proposal
!git log --oneline -3

## 3. GPU + deps check

Colab's default image already has `torch`, `torchvision`, `pandas`, `matplotlib`. This cell just confirms a GPU is attached.

In [ ]:
!nvidia-smi | head -20
import torch, torchvision, pandas, matplotlib
print('torch       :', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('torchvision :', torchvision.__version__)
print('pandas      :', pandas.__version__)
print('matplotlib  :', matplotlib.__version__)

## 4. Smoke test — 5 epochs, no-op trigger

Trains ResNet-18 on clean CIFAR-10 for 5 epochs, then evaluates `tar_acc` (clean accuracy on target class) and `asr` (with no trigger applied — should be near random ~10%).

Expected on an A100: ~3–4 min total. If Tar-ACC is reasonable and ASR is in the 5–15% range, the pipeline is sound and we can plug in real attacks.

In [ ]:
!python3 code/train_eval.py --smoke --root {WORK}/data

## 5. Render Figure 3 from current CSV

This works against `results/figure3_results.csv` — currently stub data, will be real numbers as `run_grid.py` populates it. Output goes to `figures/figure3.png` in the repo (not Drive).

In [ ]:
!python3 code/plot_figure3.py --csv results/figure3_results.csv --out figures/figure3.png
from IPython.display import Image
Image('figures/figure3.png')

## Next steps

Once the smoke test passes:

1. We add `code/attacks/badnets.py` and `code/attacks/blend.py` — each exposes a `make_trigger_fn(...)` that satisfies the `(N,3,32,32) in [0,1] -> same` contract.
2. Wrap the upstream NARCISSUS pipeline (`references/Narcissus-upstream-main.zip`) into `code/attacks/narcissus.py`.
3. Add `code/attacks/label_consistent.py` (the largest piece — uses adv perturbations).
4. Build `code/run_grid.py` — reads `results/figure3_results.csv`, skips already-completed `(method, ratio, seed)` triples, runs the rest, appends rows. Resumable across Colab session disconnects.
5. Re-render Figure 3 with real data.